# dagster-io: Resource & Chunking Playground

Test LLM, embedding, and chunking utilities interactively.

## Setup
```bash
pip install -e libs/dagster-io
# For HuggingFace local embeddings:
pip install -e 'libs/dagster-io[huggingface]'
```

Set env vars before running:
```bash
export LLM_BASE_URL=http://litellm.your-cluster:4000/v1
export LLM_API_KEY=sk-...
export LLM_MODEL=gpt-4o-mini
```

In [ ]:
import os

# Uncomment and configure as needed:
# os.environ["LLM_BASE_URL"] = "http://litellm.your-cluster:4000/v1"
# os.environ["LLM_API_KEY"] = "sk-..."
# os.environ["LLM_MODEL"] = "gpt-4o-mini"
# os.environ["EMBEDDING_PROVIDER"] = "huggingface"  # or "openai"
# os.environ["EMBEDDING_MODEL"] = "all-MiniLM-L6-v2"

## 1. Text Chunking

In [ ]:
from dagster_io import chunk_text, chunk_document, TextChunk

sample_text = """
The United States Congress passed the Infrastructure Investment and Jobs Act in 2021.
The bill allocated $1.2 trillion for infrastructure improvements across the country.

Key provisions include $110 billion for roads and bridges, $66 billion for rail,
$65 billion for broadband expansion, and $55 billion for clean water infrastructure.

The legislation was signed into law by President Biden on November 15, 2021.
It represented one of the largest infrastructure investments in decades.
""".strip()

# Test raw chunking
chunks = chunk_text(sample_text, chunk_size=200, chunk_overlap=50)
for i, c in enumerate(chunks):
    print(f"--- Chunk {i} ({len(c)} chars) ---")
    print(c)
    print()

In [ ]:
# Test document chunking with metadata
doc_chunks = chunk_document(
    document_id="test-doc-001",
    title="Infrastructure Investment and Jobs Act",
    content=sample_text,
    metadata={"source": "congress.gov", "document_type": "bill"},
    chunk_size=200,
    chunk_overlap=50,
)

for chunk in doc_chunks:
    print(f"{chunk.chunk_id} | doc={chunk.document_id} | {chunk.index}/{chunk.total_chunks}")
    print(f"  text: {chunk.text[:80]}...")
    print(f"  meta: {chunk.metadata}")
    print(f"  hash: {chunk.content_hash[:16]}...")
    print()

## 2. LLM Resource

In [ ]:
from dagster_io import LLMResource

# Create resource (normally Dagster does this)
llm = LLMResource()
llm.setup_for_execution(None)

print(f"Model: {llm.model}")
print(f"Base URL: {llm.base_url}")
print(f"Temperature: {llm.temperature}")

In [ ]:
# Test basic completion
response = llm.complete(
    "What is the capital of France?",
    system="Answer in one word."
)
print(f"Response: {response}")

In [ ]:
# Test JSON completion
import json

response = llm.complete_json(
    "List 3 European capitals.",
    system='Return a JSON object with key "capitals" containing a list of strings.'
)
print(json.dumps(json.loads(response), indent=2))

In [ ]:
# Test structured output with Pydantic
from pydantic import BaseModel, Field
from langchain_core.messages import HumanMessage, SystemMessage

class Entity(BaseModel):
    text: str = Field(description="Entity mention")
    label: str = Field(description="PERSON, ORG, GPE, DATE, LAW, EVENT")

class NERResult(BaseModel):
    entities: list[Entity]

chain = llm.with_structured_output(NERResult)
result = chain.invoke([
    SystemMessage(content="Extract named entities from the text."),
    HumanMessage(content=sample_text),
])

for ent in result.entities:
    print(f"  {ent.label}: {ent.text}")

## 3. Embedding Resource

In [ ]:
from dagster_io import EmbeddingResource

# OpenAI embeddings (or LiteLLM proxy)
emb = EmbeddingResource()
emb.setup_for_execution(None)

print(f"Provider: {emb.provider}")
print(f"Model: {emb.model}")
print(f"Base URL: {emb.base_url}")

In [ ]:
# Test embedding
texts = ["The president signed the bill.", "Congress passed legislation."]
vectors = emb.embed(texts)

print(f"Embedded {len(vectors)} texts")
print(f"Dimensions: {len(vectors[0])}")
print(f"First 5 values: {vectors[0][:5]}")

# Cosine similarity
import numpy as np
sim = np.dot(vectors[0], vectors[1]) / (np.linalg.norm(vectors[0]) * np.linalg.norm(vectors[1]))
print(f"Cosine similarity: {sim:.4f}")

In [ ]:
# Test HuggingFace local embeddings (requires dagster-io[huggingface])
# emb_hf = EmbeddingResource(provider="huggingface", model="all-MiniLM-L6-v2")
# emb_hf.setup_for_execution(None)
# vectors_hf = emb_hf.embed(texts)
# print(f"HF dimensions: {len(vectors_hf[0])}")